# មេរៀនទី 16 - ការដាក់បញ្ចូលភ្នាក់ងារដែលអាចប្រើបានជាច្រើនជាមួយ Microsoft Foundry

ក្នុងសៀវភៅកំណត់ចំណាំនេះ អ្នកកំពុងបង្កើត **ភ្នាក់ងារគាំទ្រអតិថិជនដែលទាន់សម័យសម្រាប់ផលិតកម្ម** សម្រាប់ក្រុមហ៊ុននិរូប **Contoso**។ ខុសពីមេរៀនកន្លងមក ចំណុចនៅទីនេះមិនមែនជាលំហាត់ការគិតរបស់ភ្នាក់ងារ ទេ ប៉ុន្តែគឺជារឿងទាំងអស់ដែលបង្គាប់ *ជុំវិញ* វាដែលធ្វើឲ្យភ្នាក់ងារមានសុវត្ថិភាពក្នុងការរត់នៅក្នុងអត្រាទូលំទូលាយ៖

1. **ការហៅឧបករណ៍** — ស្វែងរកលំអិតនិងបង្កើតសំបុត្រជំនួយ។
2. **RAG** — ចម្លើយគោលនយោបាយពីមូលដ្ឋានចំណេះដឹង។ 
3. **ការចងចាំ** — ការចងចាំអតិថិជនឲ្យបានឆ្លងកាត់ជើងហោះហើរ។
4. **ការបញ្ជូនម៉ូដែល** — ផ្ញើសំណើសាមញ្ញទៅម៉ូដែលតូច និងសំណើស្មុគស្មាញទៅម៉ូដែលធំ។
5. **ការផ្ទុកចម្លើយ** — ផ្តល់សេវាសំណួរដដែលដោយមិនចាំបាច់ហៅម៉ូដែលឡើងវិញ។
6. **ការអនុម័តដោយមនុស្ស** — ការសងប្រាក់លើសចំនុចកំណត់ទំរង់ទាមទារ ឲ្យរង់ចាំការផ្ទៀងផ្ទាត់។
7. **កម្រិតវាយតម្លៃ** — ការប្រឡងម៉ាស៊ីនក្រៅបណ្ដាញដែលទប់ស្កាត់ការបញ្ចេញផ្សាយអាក្រក់។
8. **ការតាមដាននិងមើលសភាព** — ការតាមដាន OpenTelemetry ជុំវិញសំណើរ প্রত্যেক។

ផ្នែកនីមួយៗមានលក្ខណៈឯករាជ្យនិងអាចរត់បាន។ សូមអានគ្រប់បន្ទាត់ — ទ្រទ្រង់ផលិតកម្មត្រូវបានរក្សាទុកក្នុងទំហំនីឡើងយ៉ាងមានគោលបំណង។


## ការតំឡើង

មុនពេលរត់សៀវភៅចំណាំនេះ សូមប្រាកដថាអ្នកមាន៖

1. **គម្រោង Microsoft Foundry** មួយដែលមានម៉ូដែលសន្ទនា​បានបញ្ចេញ (ឧ. `gpt-4.1-mini`)។
2. **បានចូលគណនីជាមួយ Azure CLI** — រត់ `az login` នៅក្នុងបន្ទាត់បញ្ជារបស់អ្នក។
3. **កំណត់អថេរបរិស្ថានដែលត្រូវការ៖**
   - `AZURE_AI_PROJECT_ENDPOINT` — ចំណុចបញ្ចប់គម្រោង Microsoft Foundry របស់អ្នក។
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ឈ្មោះម៉ូដែលដែលបានបញ្ចេញរបស់អ្នក។

ផ្នែក RAG ប្រើ **Azure AI Search** ពេលដែល `AZURE_SEARCH_SERVICE_ENDPOINT` និង `AZURE_SEARCH_API_KEY` ត្រូវបានកំណត់ ហើយបរាជ័យទៅកាន់ការស្វែងរកក្នុងម៉ាស៊ីនដើម្បីឲ្យសៀវភៅចំណាំរត់ដោយមិនមានធនធាន Search។


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## ១. ឧបករណ៍

ឧបករណ៍ផលិតកម្មធ្វើការងារពិតប្រឆាំងនឹងប្រព័ន្ធពិត។ នៅទីនេះយើងហាក់ដូចជាទិន្នន័យបញ្ជាទិញមួយនិងប្រព័ន្ធសំបុត្រដែលមានមុខងារពី Python របស់ម៉ាស៊ីនធម្មតា។ `@tool` decorator បង្ហាញពួកវាទៅអ្នកមេត្តា។

សូមទស្សនាថា `issue_refund` ប្រើ `approval_mode="always_require"` សម្រាប់ការបញ្ចូនប្រាក់វិញលើសចំនួនជាក់លាក់ — នេះគឺជាការដំណើរការក្នុងមនុស្ស-ក្នុង-រង្វង់ដែលយើងប្រើប្រាស់បន្ទាប់មក។


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — ឃ្លាំងចំណេះដឹងគោលនយោបាយ

សំណួរក្នុងគោលនយោបាយ ("រយៈពេលបង្វិលទំនិញរបស់អ្នកយ៉ាងដូចម្តេច?") គួរត្រូវបានឆ្លើយតបពីប្រភពដែលមានអំណាច ដែលមិនមែនជាចំណាំគំនិតរបស់ម៉ូដែល។ យើងបត់បែនឃ្លាំងចំណេះដឹងតូចមួយជាឧបករណ៍ស្វែងរក។

ក្នុងការផលិតនេះគឺជា **Azure AI Search**; នៅទីនេះយើងផ្ដល់នូវការស្វែងរកពាក្យគន្លឹះក្នុងចំណាំដើម្បីឱ្យសៀវភៅកំណត់ត្រារត់បាននៅគ្រប់កន្លែង ហើយបម្លាស់ទៅ Azure AI Search ដោយស្វ័យប្រវត្តិពេលដែលអក្សរបរិស្ថានមានរួចហើយ។


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. ម៉ែម៉ូរី

អ្នកគាំទ្រម្នាក់ដែលភ្លេចថា​និយាយជាមួយនរណាគឺ​ជា​អ្នកគាំទ្រដែលអ្វីមិនល្អ។ យើងរក្សាទុកហាងប្រវត្តិរូបតូចមួយសម្រាប់អតិថិជនម្នាក់ៗ ហើយបញ្ចូលសេចក្ដីសង្ខេបខ្លីទៅក្នុងការណែនាំរបស់ភ្នាក់ងារនោះ។ ក្នុងប្រព័ន្ធផលិតនេះគឺជាសេវាម៉ែម៉ូរី (មើលមេរៀនទី 13)។ នៅទីនេះ dict មួយធ្វើឱ្យគំរូនេះប្រាកដរបស់។


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. ការបញ្ជូនគំរូ និងការផ្ទុកចម្លើយស្តុក

ឧបករណ៍ចំណាយពីរដាក់ក្នុងអ្នកដំណើរការសំណើតែមួយ៖

- **ការបញ្ជូន**៖ កម្មវិធីជ្រើសរើសដោយវិធីសាស្ត្រសម្រួលថ្មោងថាតើសំណើត្រូវការគំរូតូចឬធំ។
- **ការផ្ទុកចម្លើយ**៖ សំណួរដែលបានធ្វើបរិចម្រាញ់ត្រូវបានផ្តល់ពីស្តុកដោយផ្ទាល់ដោយគ្មានការហៅគំរូឡើយ។

កម្មវិធីជ្រើសរើសនៅទីនេះគឺមានភាពសាមញ្ញដោយចេតនា។ Trong sản xuất, bạn sẽ xác thực nó với lưu lượng và có thể thay thế bằng Model Router của Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. តួភាគី តំណលំនឹងមនុស្ស និងការត្រួតពិនិត្យ

ឥឡូវនេះយើងប្រមូលតួភាគីពីឧបករណ៍ខាងលើ ហើយបង្ហាញសំណើរ​រៀងៗគ្នា​ក្នុងស្វាន OpenTelemetry។ មុខងារ `handle_support_request` គឺជាគ្រប់គ្រងសំណើក្នុងផលិតកម្ម៖ cache → route → trace → run → cache។

ការអនុម័តពីមនុស្ស ត្រូវបានគ្រប់គ្រងដោយស៊ុមវ៉ាក์: ពីព្រោះ `issue_refund` មាន `approval_mode="always_require"` គឺដំណើរការពេលវេលារងចាំ ហើយបង្ហាញសំណើអនុម័តដែលយើងដោះស្រាយយ៉ាងច្បាស់។


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. វេទ្វប្រមាណលទ្ធផល

នេះគឺជាវេទ្វចេញផ្សាយពីមេរៀន៖ សំណុំតេស្តអ៊ីនឥណ្ឌOffline មួយវាស់បញ្ចាក់ភ្នាក់ងារនិងការចាក់ផ្សាយត្រូវបន្តបើអត្រាការឆ្លងកាត់លើសតុល្យភាព។ អ្នកវាស់នៅទីនេះគឺជាការត្រួតពិនិត្យការឈ្លោះពាក្យគន្លឹះដោយសាមញ្ញដើម្បីរក្សាឲ្យសៀវភៅកំណត់ត្រានេះមានខ្លួនឯង; ក្នុងការផលិតអ្នកនឹងប្រើ LLM-ជាអ្នកវាយតម្លៃ ឬប្រព័ន្ធវាយតម្លៃ (មើលមេរៀនទី ១០)។


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## ដើម្បីបញ្ចូលគ្នា៖ ការចេញផ្សាយដែលបានសំលាញ់

កោសិលខាងក្រោមបង្ហាញពីល៊ុនពេញលេញដែលមេរៀនពិពណ៌នា៖ បើកទ្វារវាយតម្លៃ ហើយ "ចេញផ្សាយ" តែបើវាដើរត្រឹមត្រូវ។ នេះជារូបមន្តដែលអ្នកនឹងរត់នៅក្នុង CI មុនពេលធ្វើការផ្សព្វផ្សាយកំណែភ្នាក់ងារមួយទៅសេវា Foundry Agent។


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## សេចក្តីសង្ខេប

អ្នកបានបង្កើតភ្នាក់ងារគាំទ្រអតិថិជនដែលរួមមានការប្រតិបត្តិទាំងអស់ក្នុងស្ថានភាពផលិតកម្ម:

- **ឧបករណ៍, RAG, និងកម្មវិធីចងចាំ** ផ្តល់ឱ្យភ្នាក់ងារជំនាញនិងបរិបទ។
- **ការបញ្ជូនម៉ូដែលនិងការបម្រើបានដាក់ក្នុងចង្រៃ** រក្សាពេលយឺតនិងថ្លៃដើមនៅក្រោមការត្រួតពិនិត្យ។
- **ការអនុម័តដោយមនុស្ស** បានការពារ​សកម្មភាពដែលមានហានិភ័យខ្ពស់ដូចជាការសងប្រាក់វិញច្រើន។
- **ក្រវាំងប៉ាន់ប្រមាណ** បិទបញ្ញើការចេញផ្សាយខុសស្រប មុនពេលវាផ្សាយចេញ។
- **ការតាមដាន** ធ្វើឲ្យគ្រប់សំណើអាចមើលឃើញបាន។

### ប្រឈមមុខ

ពង្រីកភ្នាក់ងារនេះដើម្បី:

1. **គាំទ្រម៉ូដែលច្រើន** — បន្ថែមផ្នែក "ហេតុផល" ទីបី ហើយបញ្ជូនសំណើរ/បណ្តឹងទៅវា។
2. **បន្ថែមក្រវាំងប៉ាន់ប្រមាណ** — ពង្រីក `TEST_CASES` ដើម្បីរួមបញ្ចូលសេណារីយ៉ូអនុម័តសងប្រាក់វិញ ហើយបញ្ជាក់ថាក្រវាំងចាប់សកម្មភាព regressions បាន។
3. **បន្ថែមការផ្លូវចំណាយជាភាពយល់** — តាមដានថ្លៃដើមដែលគិតបានក្នុងមួយសំណើ (តិច ទឹកចិត្ត ឬ cache) ហើយបោះពុម្ពរបាយការណ៍ថ្លៃតម្លៃបន្ទាប់ពីកម្រងសំណួររួមគ្នា។

ក្នុងមេរៀនបន្ទាប់ អ្នកនឹងធ្វើដំណើរបញ្ច្រាស់ផ្នែកអាគារយោបល់ និងរត់ភ្នាក់ងារជាពេញលេញលើម៉ាស៊ីនផ្ទាល់របស់អ្នកជាមួយ Microsoft Foundry Local និង Qwen។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
